# Generate: Multiple Scenario PartMC vs. MAM4

This notebook walks through using AMBRS to run box-model simulations of PartMC and MAM4 under several sets of consistent conditions.
The data generated using this notebook will be saved to `tutorial/data/prepared/multiple_scenario_partmc_mam4` and can be analyzed using the tutorial notebook at `tutorial/notebooks/02_plot_results/02_multiple_scenario_partmc_mam4.ipynb`.

By the end of this tutorial, you should be able to:
* Understand how to configure AMBRS for comparisons of PartMC and MAM4
* Use AMBRS to run box-model simulations of PartMC and MAM4
* Generate diagnostic data for comparing PartMC and MAM4 results
* Continue to `tutorial/notebooks/02_plot_results/02_multiple_scenario_partmc_mam4.ipynb`

<small>__Note:__ This note assumes that PartMC, MAM4 and AMBRS are properly built and accessible. If you have not already done so, follow the instructions in the `README` under "Running Tutorial Notebooks" before proceeding.</small>

## AMBRS Setup

First, we'll pull in the dependencies we'll need for running AMBRS.

In [ ]:
import os
import pickle
import shutil
import subprocess
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass
from math import log10, ceil
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import ambrs

from part2pop.population import build_population
from part2pop.viz.builder import build_plotter

Next, we'll set up some paths we'll use for running the models and managing data files.

<small>__Note:__ These paths assume the working directory is the one that includes this notebook.
If you run this notebook from a different working directory, update `tutorial_root` accordingly. </small>

In [ ]:
tutorial_root = Path.cwd().parent.parent

# raw model output will be saved here
raw_output_dir = (
    tutorial_root
    / 'data'
    / 'raw'
    / 'multiple_scenario_partmc_mam4'
)

# post-processed diagnostic data will be saved here
prepared_output_dir = (
    tutorial_root
    / 'data'
    / 'prepared'
    / 'multiple_scenario_partmc_mam4'
)

# Set the AMBRS_MODEL_DIR to the path where the model binaries exist.
# If this variable is not specified, the models should be in a folder included in
# the working directory.
model_dir_str = os.environ.get('AMBRS_MODEL_DIR')
model_dir = Path(model_dir_str).resolve() if model_dir_str else None

# clear existing output if necessary
if raw_output_dir.exists():
    shutil.rmtree(raw_output_dir)
if prepared_output_dir.exists():
    shutil.rmtree(prepared_output_dir)
raw_output_dir.mkdir(parents=True)
prepared_output_dir.mkdir(parents=True)

print(f'Raw output directory: {raw_output_dir}')
print(f'Prepared output directory: {prepared_output_dir}')
print(f'Model executable directory: {model_dir}')

<small>__NOTE:__ If you followed the instructions in the `README` for running the tutorial notebooks, the model executable directory will be `None`. PartMC and MAM4 executables will be on the `PATH` in the Jupyter kernel's virtual environment.</small>

Lastly, let's find the executables for PartMC and MAM4

In [ ]:
if model_dir:
    partmc_exe = model_dir / 'partmc'
    mam4_exe = model_dir / 'mam4'
else:
    partmc_path = shutil.which('partmc')
    mam4_path = shutil.which('mam4')
    partmc_exe = Path(partmc_path) if partmc_path else None
    mam4_exe = Path(mam4_path) if mam4_path else None

if not partmc_exe or not partmc_exe.is_file():
    raise RuntimeError(
        'PartMC executable not found. '
        'Build/install PartMC or set environment variable AMBRS_MODEL_DIR to folder containing it.'
    )
if not mam4_exe or not mam4_exe.is_file():
    raise RuntimeError(
        'MAM4 executable not found. '
        'Build/install MAM4 or set environment variable AMBRS_MODEL_DIR to folder containing it.'
    )

## Build AMBRS Scenarios

Now, let's create some scenarios for our comparison. To keep things simple, we'll use versions of PartMC and MAM4 that are internally fixed to the same set of chemical species and reactions. Our scenarios will differ only in the initial and evolving conditions of the simulations.

First, let's create the components that will be shared across all the scenarios.

In [ ]:
# aerosol species
aerosol_species = (
    so4 := ambrs.AerosolSpecies(
        'SO4', 
        molar_mass=96.0, 
        density=1800.0, 
        hygroscopicity=0.65,
    ),
    nh4 := ambrs.AerosolSpecies(
        'NH4', 
        molar_mass=18.0, 
        density=1800.0, 
        hygroscopicity=0.65,
    ),
    pom := ambrs.AerosolSpecies(
        'OC', molar_mass=12.01, 
        density=1000.0, 
        hygroscopicity=0.001, 
        aliases='POM',
    ),
    soa := ambrs.AerosolSpecies(
        'MSA',
        molar_mass=40.0, 
        density=2600.0, 
        hygroscopicity=0.53, 
        aliases='SOA',
    ),
    bc  := ambrs.AerosolSpecies(
        'BC',  
        molar_mass=12.01, 
        density=1800.0, 
        hygroscopicity=0.0,
    ),
    dst := ambrs.AerosolSpecies(
        'OIN', 
        molar_mass=135.065, 
        density=2600.0, 
        hygroscopicity=0.1, 
        aliases='DST',
    ),
    na  := ambrs.AerosolSpecies(
        'Na',  
        molar_mass=23.0, 
        density=2200.0, 
        hygroscopicity=0.53, 
        aliases='NCL',
    ),
    cl  := ambrs.AerosolSpecies(
        'Cl',  
        molar_mass=35.5, 
        density=2200.0, 
        hygroscopicity=0.53, 
        aliases='NCL',
    ),
    ncl := ambrs.AerosolSpecies(
        'Na',  
        molar_mass=23.0,
        density=2200.0, 
        hygroscopicity=0.53,
        aliases='NCL',
    ),
    h2o := ambrs.AerosolSpecies(
        'H2O', 
        molar_mass=18.0, 
        density=1000.0, 
        ions_in_soln=1,
    ),
)

# gas-phase species
gas_species = (
    ambrs.gas.GasSpecies(name='SO2', molar_mass=64.07),
    ambrs.gas.GasSpecies(name='O3', molar_mass=48.00),
    ambrs.gas.GasSpecies(name='H2SO4', molar_mass=98.079),
)

print(f'Total aerosol species: {len(aerosol_species)}')
print(f'Total gas species: {len(gas_species)}')

Next up is defining the aerosol state. MAM4 uses an aerosol representation that consists of four modes, each with specific chemical species that can be present. We need to define our initial aerosol population using these modes. To keep things simple, we'll just populate the accumulation mode for our simulations.

In [ ]:
geometric_mean_diameters = [1e-7, 2.6e-8, 2e-6, 1e-8]
geometric_standard_deviations = [log10(1.8), log10(1.6), log10(1.6), log10(1.8)]
small = 1e-30

def define_aerosol_state(config: dict[str, Any]) -> ambrs.aerosol.AerosolModalSizeState:
    """Define the initial aerosol population using MAM4 modes."""
    accumulation= ambrs.aerosol.AerosolModeState(
        name='accumulation',
        species=[so4, nh4, pom, soa, bc, dst, ncl, h2o],
        number=config["aerosol number concentration"],
        geom_mean_diam=geometric_mean_diameters[0],
        log10_geom_std_dev=geometric_standard_deviations[0],
        mass_fractions=[config['so4'], small, config['pom'], small, small, small, small, small],
    )
    aiken = ambrs.aerosol.AerosolModeState(
        name='aiken',
        species=[so4, soa, ncl],
        number=small,
        geom_mean_diam=geometric_mean_diameters[1],
        log10_geom_std_dev=geometric_standard_deviations[1],
        mass_fractions=[1, small, small],
    )
    coarse = ambrs.aerosol.AerosolModeState(
        name='coarse',
        species=[dst, ncl, so4, bc, pom, soa],
        number=small,
        geom_mean_diam=geometric_mean_diameters[2],
        log10_geom_std_dev=geometric_standard_deviations[2],
        mass_fractions=[small, small, 1, small, small, small, small],
    )
    primary_carbon = ambrs.aerosol.AerosolModeState(
        name='primary carbon',
        species=[pom, bc],
        number=small,
        geom_mean_diam=geometric_mean_diameters[3],
        log10_geom_std_dev=geometric_standard_deviations[3],
        mass_fractions=[1, small],
    )
    return ambrs.aerosol.AerosolModalSizeState(
        modes=(accumulation, aiken, coarse, primary_carbon)
    )

Next, we'll define sets of conditions that vary across scenarios and a helper function that uses them to build full AMBRS scenarios.

In [ ]:
scenario_options = {
    'base-case': {
        'gas mixing ratios': (0, 0, 1e-5),  # SO2, O3, H2SO4 (mol mol-1)
        "aerosol number concentration": 5e12,  # (# m-3)
        'so4': 0.45,  # (kg kg-1)
        'pom': 0.55,  # (kg kg-1) 
    },
    'high-sulfuric-acid': {
        'gas mixing ratios': (0, 0, 1e-3),  # SO2, O3, H2SO4 (mol mol-1)
        'so4': 0.45,  # (kg kg-1)
        'aerosol number concentration': 5e12,  # (# m-3)
        'pom': 0.55,  # (kg kg-1) 
    },
    'high-particle-count': {
        'gas mixing ratios': (0, 0, 1e-5),  # SO2, O3, H2SO4 (mol mol-1)
        'aerosol number concentration': 5e14,  # (# m-3)
        'so4': 0.45,  # (kg kg-1)
        'pom': 0.55,  # (kg kg-1)
    }
}

temperature = 298.0
pressure = 101325

def generate_scenario(config: dict[str, Any]) -> ambrs.scenario.Scenario:
    """Generate a full scenario with the specified conditions."""

    return ambrs.scenario.Scenario(
        aerosols=aerosol_species,
        gases=gas_species,
        size=define_aerosol_state(config),
        gas_concs=config['gas mixing ratios'],
        flux=0,
        relative_humidity=0.0,
        temperature=temperature,
        pressure=pressure,
        height=500.0,
    )

scenarios = {
    key: generate_scenario(val) for key, val in scenario_options.items()
}

print(f'Total scenarios: {len(scenarios)}')

## Configure the Models

Before we run the models, we'll need to configure them for the comparison we want to perform. To isolate structural errors, we'll want to make sure that the configuration of PartMC and MAM4 is as similar as possible.

Let's define some variables that we'll use for both models. We'll run three configuration with each of our scenarios:
* condensation only
* coagulation only
* condensation with coagulation

In [ ]:
# simulation timing
dt = 60  # time step (s)
nstep = 60  # number of time steps (#) = 1 hr

# physical processes to include in the simulations
model_configs = {
    'condensation-only': ambrs.aerosol.AerosolProcesses(
        coagulation=False,
        condensation=True,
    ),
    'coagulation-only': ambrs.aerosol.AerosolProcesses(
        coagulation=True,
        condensation=False,
    ),
    'condensation-with-coagulation': ambrs.aerosol.AerosolProcesses(
        coagulation=True,
        condensation=True,
    ),
}

### PartMC

Next, let's configure PartMC for each combination of processes. Here we'll also set configuration options that are specific to PartMC.

In [ ]:
n_part = 5000  # number of computational particles

partmc_models = {
    'condensation-only': ambrs.partmc.AerosolModel(
        processes=model_configs["condensation-only"],
        run_type='particle',
        n_part=n_part,
        n_repeat=1,
    ),
    'coagulation-only': ambrs.partmc.AerosolModel(
        processes=model_configs["coagulation-only"],
        run_type='particle',
        n_part=n_part,
        n_repeat=1,
    ),
    'condensation-with-coagulation': ambrs.partmc.AerosolModel(
        processes=model_configs["condensation-with-coagulation"],
        run_type='particle',
        n_part=n_part,
        n_repeat=1,
    ),
}

### MAM4

Let's do the same thing for MAM4. Notice that there are no MAM4-specific options, so we just need to specify the processes to include.

In [ ]:
mam4_models = {
    'condensation-only': ambrs.mam4.AerosolModel(
        processes=model_configs["condensation-only"],
    ),
    'coagulation-only': ambrs.mam4.AerosolModel(
        processes=model_configs["coagulation-only"],
    ),
    'condensation-with-coagulation': ambrs.mam4.AerosolModel(
        processes=model_configs["condensation-with-coagulation"],
    ),
}

## Run the Simulations

At this point, we have everything in place to run the models. Let's create a helper function for kicking off each run.

In [ ]:
def run_model(
    prefix: str,
    executable: Path,
    model: ambrs.partmc.AerosolModel,
    scenario: ambrs.scenario.Scenario
) -> None:
    """Run a model with the scenario provided."""
    input = model.create_input(
        scenario=scenario,
        dt=dt,
        nstep=nstep,
    )

    # create a dedicated folder for this run using the prefix
    run_dir = raw_output_dir / model.name / prefix
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True)

    model.write_input_files(
        input=input,
        dir=run_dir,
        prefix=prefix,
    )

    result = subprocess.run(
        [item for item in model.invocation(exe=executable, prefix=prefix).split(' ')],
        cwd=run_dir,
        capture_output=True,
        text=True,
        check=True,
    )

    print(f'{model.name} run {prefix} completed.')
    print(result.stdout[-1000:])

Instead of running the models now, we'll first get prepared to calculate the diagnostics we'll use the in the AMBRS comparisons.
Since PartMC output can be enormous, we'll run a scenario, calculate and save the diagnostics, then delete the run folder before moving on to the next scenario.

## Calculate Diagnostics

For the diagnostics, we're going to use the `part2pop` package that can plot a variety of diagnostic properties of aerosol systems in several popular aerosol representations. You can learn more about part2pop here: https://github.com/pnnl/part2pop.

First, let's create separate helper functions for extracting the PartMC and MAM4 states from their respective output. Let's start with PartMC. 

In [ ]:
def get_partmc_states(prefix: str) -> dict[str, Any]:
    """Return the output aerosol states for each timestep of a given PartMC run."""
    run_dir = raw_output_dir / 'partmc' / prefix
    files = sorted((run_dir / 'out').glob('*.nc'))
    if not files:
        raise RuntimeError(f'No PartMC files found for run: {prefix}')

    def extract_timestep(path: str) -> int:
        return int(path.stem.split('_')[-1])

    timesteps = [extract_timestep(path) for path in files]

    return [
        {
            'population': build_population({
                'type': 'partmc',
                'partmc_dir': run_dir,
                'timestep': timestep,
                'repeat': 1,
            }),
            'model': 'PartMC',
            'scenario': prefix,
            'timestep': timestep,
            'time_min': (timestep - 1) * dt / 60.0,
        }
        for timestep in timesteps
    ]

And, now MAM4.

In [ ]:
def get_mam4_states(prefix: str, scenario: ambrs.Scenario) -> dict[str, Any]:
    """Return the output aerosol states for each timestep of a given MAM4 run."""
    run_dir = raw_output_dir / 'mam4' / prefix

    mam4_ds = xr.open_dataset(run_dir / 'mam_output.nc')
    mam4_nsteps = mam4_ds['nsteps'].values

    # note: timesteps are expected to start at 1 (initial conditions)
    return [
        {
            'population': build_population({
                'type': 'mam4',
                'mam4_dir': run_dir,
                'timestep': 1,
                'D_is_wet': False,
                'GMD_init': geometric_mean_diameters,
                'GSD': [ 10**item for item in geometric_standard_deviations ],
                'N_sigmas': 10,
                'N_bins': 2000,
            }),
            'model': 'MAM4',
            'scenario': prefix,
            'timestep': 1,
            'time_min': 0,
        },
        *[
            {
                'population': build_population({
                    'type': 'mam4',
                    'mam4_dir': run_dir,
                    'timestep':timestep + 2,
                    'D_is_wet': False,
                    'GSD': [ 10**item for item in geometric_standard_deviations ],
                    'N_sigmas': 10,
                    'N_bins': 2000,
                    'T': temperature,
                    'p': pressure,
                }),
                'model': 'MAM4',
                'scenario': prefix,
                'timestep': timestep + 2,
                'time_min': (timestep + 1) * dt / 60.0,
            }
            for timestep in mam4_nsteps
        ]
    ]

Next, we need to determine which diagnostics we would like to include in the comparison. For this tutorial, we'll choose:
* total number concentration
* geometric mean diameter
* geometric standard deviation
* cloud condensation nucleii
* total optical obsorption

You can see all the diagnostics available in `part2pop` [here](https://github.com/pnnl/part2pop)

In [ ]:
plotter_configs = [
    {
        'varname': 'number_concentration',
        'x': 'time_min',
        'series': 'model',
        'population': 'population',
        'legend': True,
        'sort': True,
        'plotter_type': 'series_line',
    },
    {
        'varname': 'geom_mean_diameter',
        'var_cfg': {
            'wetsize': False,
        },
        'x': 'time_min',
        'yscale': 'log',
        'series': 'model',
        'population': 'population',
        'legend': True,
        'sort': True,
        'plotter_type': 'series_line',
    },
    {
        'varname': 'geom_std_dev',
        'var_cfg': {
            'wetsize': False,
        },
        'x': 'time_min',
        'series': 'model',
        'population': 'population',
        'legend': True,
        'sort': True,
        'plotter_type': 'series_line',
    },
    {
        'varname': 'dNdlnD',
        'var_cfg': {
            'diam_grid': np.logspace(-9, -5, 80),
            'D_min': 1e-9,
            'D_max': 2e-5,
        },
        'series': 'model',
        'population': 'population',
        'legend': True,
        'plotter_type': 'state_line',
    },
    {
        'varname': 'frac_ccn',
        'var_cfg': {
            's_grid': np.logspace(-2, 1, 100),
            'T': 298.15,
        },
        'series': 'model',
        'population': 'population',
        'legend': True,
        'sort': True,
        'plotter_type': 'state_line',
    },
    {
        'varname': 'b_scat',
        'var_cfg': {
            'morphology': 'homogeneous',
            'wvl_grid': np.linspace(350e-9,1050e-9,29),
            'rh_grid': [0]
        },
        'series': 'model',
        'population': 'population',
        'legend': True,
        'sort': True,
        'plotter_type': 'state_line',
    },
]

def calculate_and_pickle(
        config: dict[str, Any], 
        prefix: str, 
        partmc_states: dict[str, Any], 
        mam4_states: dict[str, Any]
    ) -> None:
    """Assemble and pickle the comparison data for future plotting."""
    plotter = build_plotter(config['plotter_type'], config)
    
    if config['plotter_type'] == 'series_line':
        # plot a time series of a scalar property
        prepared = [{
            'partmc': plotter.prep(partmc_states),
            'mam4': plotter.prep(mam4_states),
        }]
    elif config['plotter_type'] == 'state_line':
        # plot a property for a single timestep
        prepared = [
            {
                'partmc': plotter.prep(partmc_states[index]['population']),
                'mam4': plotter.prep(mam4_states[index]['population']),
                'step': index,
            }
            for index in np.linspace(0, len(mam4_states) - 1, num=6, dtype=int)
        ]
    
    payload = [
        {
            'name': f'{prefix}_partmc_vs_mam4_{config['varname']}',
            'plotter_type': config['plotter_type'],
            'plotter_config': config,
            'prepared': {
                'partmc': prep['partmc'],
                'mam4': prep['mam4'],
            },
            'metadata': {
                'workflow': '02_multiple_scenario_partmc_mam4',
                'case': prefix,
                'models': [
                    'PartMC',
                    'MAM4',
                ],
                'dt': dt,
                **(
                    {
                        'filename': f'{prefix}-{config["varname"]}-{prep["step"]}.pkl',
                        'step': prep['step'],
                        'time': prep['step'] * dt,
                    } if 'step' in prep else {
                        'filename': f'{prefix}-{config["varname"]}.pkl',
                    }
                ),
                'n_part': n_part,
                'timesteps': nstep,
                'source_run_dirs': [
                    str(raw_output_dir / 'partmc' / prefix),
                    str(raw_output_dir / 'mam4' / prefix),
                ]
            }   
        }
        for prep in prepared
    ]
    for item in payload:
        with (prepared_output_dir / item['metadata']['filename']).open('wb') as f:
            pickle.dump(item, f)


## Putting it all together

We're at the last step! Let's run PartMC and MAM4 for each combination of model configuration and scenario, and assemble and save the diagnostic output.

In [ ]:
def run_ambrs_case(config_name, scenario_name, scenario) -> str:
    prefix = f"{config_name}-{scenario_name}"

    # run the models
    run_model(prefix, partmc_exe, partmc_models[config_name], scenario)
    run_model(prefix, mam4_exe, mam4_models[config_name], scenario)
    
    # load model output
    mam4_states   = get_mam4_states(prefix, scenario)
    partmc_states = get_partmc_states(prefix)

    # assemble and save the diagnostic data
    for config in plotter_configs:
        calculate_and_pickle(config, prefix, partmc_states, mam4_states)

    return f'{prefix} complete!'

if __name__ == '__main__':
    input_config_names = []
    input_scenario_names = []
    input_scenarios = []
    for config_name in model_configs:
        for scenario_name, scenario in scenarios.items():
            input_config_names.append(config_name)
            input_scenario_names.append(scenario_name)
            input_scenarios.append(scenario)

    with ProcessPoolExecutor() as executor:
        results = executor.map(
            run_ambrs_case,
            input_config_names,
            input_scenario_names,
            input_scenarios
        )

    for r in results:
        print(r)

Let's test by loading a single `.pkl` file and plotting the comparison.

In [ ]:
with (prepared_output_dir / 'condensation-with-coagulation-base-case-dNdlnD-0.pkl').open('rb') as f:
    loaded_payload = pickle.load(f)

loaded_plotter = build_plotter(
    loaded_payload['plotter_type'],
    loaded_payload['plotter_config'],
)

fig, ax = plt.subplots(figsize=(6, 4))
loaded_plotter.plot(ax=ax, prepared=loaded_payload['prepared']['partmc'])
loaded_plotter.plot(ax=ax, prepared=loaded_payload['prepared']['mam4'])
fig.tight_layout()

That's it! You're ready to move on to the next tutorial: `tutorials/notebooks/02_plot_results/02_multiple_scenarios_partmc_mam4.ipynb`. There you will use the data you just saved to plot the diagnostics and investigate the differences in structural errors between PartMC and MAM4.